# 02: Preprocessing

Cleans the raw data and produces the train/test split used by every notebook
downstream. Findings/evidence behind these decisions live in `01_eda.ipynb`.

**Key discipline followed throughout:** any statistic *learned* from data
(a median, a winsorization cap) is computed from `X_train` only, then
applied to both `X_train` and `X_test`, never the other way around. Doing
the split before computing these statistics (rather than cleaning the full
dataset and splitting after) avoids leaking test-set information into
training-data preprocessing.

**Output:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` saved to
`../data/processed/`, consumed by `03_modeling.ipynb`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/raw/cs-training.csv')
print(df.shape)

(150000, 12)


## Fixed-rule cleaning (safe to apply before the split)

These don't depend on any learned statistic, so order relative to the split
doesn't matter for leakage, but the `income_was_missing` flag **must** be
created before `MonthlyIncome` is imputed, otherwise the "was it missing"
information is destroyed.

In [3]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df['income_was_missing'] = df['MonthlyIncome'].isnull().astype(int)

## Train/test split

Stratified on the target to preserve the ~6.7% positive rate in both sets.

In [4]:
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print(y_train.value_counts(normalize=True))

X_train: (120000, 11)
X_test: (30000, 11)
SeriousDlqin2yrs
0    0.933158
1    0.066842
Name: proportion, dtype: float64


## Learn-from-train-only preprocessing

Sentinel-code replacement (96/98 -> 0) is a fixed rule, applied to both sets
directly. Medians and winsorization caps are *learned from `X_train` only*,
then applied to both `X_train` and `X_test`. This is the fit/transform
discipline that avoids leakage.

In [5]:
age_median = X_train['age'].replace(0, pd.NA).median()
income_median = X_train['MonthlyIncome'].median()
dependents_median = X_train['NumberOfDependents'].median()
revol_cap = X_train['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99)
debt_cap = X_train['DebtRatio'].quantile(0.99)

print(f"age_median={age_median}, income_median={income_median}, "
      f"dependents_median={dependents_median}")
print(f"revol_cap={revol_cap:.3f}, debt_cap={debt_cap:.3f}")

past_due_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

for col in past_due_cols:
    X_train[col] = X_train[col].replace([96, 98], 0)
    X_test[col] = X_test[col].replace([96, 98], 0)

for X_split in (X_train, X_test):
    X_split['age'] = X_split['age'].replace(0, age_median)
    X_split['MonthlyIncome'] = X_split['MonthlyIncome'].fillna(income_median)
    X_split['NumberOfDependents'] = X_split['NumberOfDependents'].fillna(dependents_median)
    X_split['RevolvingUtilizationOfUnsecuredLines'] = X_split['RevolvingUtilizationOfUnsecuredLines'].clip(upper=revol_cap)
    X_split['DebtRatio'] = X_split['DebtRatio'].clip(upper=debt_cap)

age_median=52.0, income_median=5390.0, dependents_median=0.0
revol_cap=1.094, debt_cap=4977.000


In [6]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
income_was_missing                      0
dtype: int64
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
income_was_missing                      0
dtype: int64


## Save processed data

Saved with index preserved, so rows stay traceable back to the original
`cs-training.csv` if ever needed.

In [7]:
X_train.to_csv('../data/processed/X_train.csv', index=True)
X_test.to_csv('../data/processed/X_test.csv', index=True)
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv', index=True)

print("Saved processed data to ../data/processed/")

Saved processed data to ../data/processed/
